# Removing the GNSS PPS sawtooth with u-blox qErr

A GNSS timing receiver can only start its PPS on an edge of its internal clock, so every pulse is
early or late by up to half a clock period (±4 ns for an 8 ns clock). Because the internal clock
runs at a slightly different rate from GNSS time, that error drifts in a characteristic
**sawtooth**. The receiver knows the error of the next pulse in advance and reports it as `qErr` in
the `UBX-TIM-TP` message. Applying it to a time-interval-counter (TIC) measurement of the PPS removes
the sawtooth, and the short-term stability of the measurement improves.

This notebook does that with ntpstats 3.4+ on a synthetic 30-minute example: a receiver log in UBX
(mixed with NMEA, as receivers write it) and a TIC measuring the receiver's PPS against an OCXO.

In [ ]:
from pathlib import Path

import numpy as np

from ntpstats import api as nt
from ntpstats.ubx import apply_qerr

DATA = next(p for p in (Path("../data"), Path("examples/data"), Path("../../examples/data")) if p.is_dir())
fmt = nt.format_seconds
receiver = {s.meta["message"]: s for s in nt.load(str(DATA / "ubx-timing.ubx"))}
tic = nt.load_one(str(DATA / "pps-tic.csv"))
qerr = receiver["UBX-TIM-TP"]
print(f"TIC: {len(tic)} PPS measurements; receiver: {len(qerr)} TIM-TP and "
      f"{len(receiver['UBX-NAV-CLOCK'])} NAV-CLOCK messages")
print(f"qErr spans {fmt(qerr.offset.min())} .. {fmt(qerr.offset.max())}")

## Applying qErr

Each PPS sample is paired with the `qErr` of the pulse nearest in time. u-blox does not state the
sign with which `qErr` applies, so ntpstats tries both and keeps the one that removes the sawtooth
(it is the fastest component, so the right sign gives the smaller sample-to-sample variance); the
choice is reported and can be fixed with `sign=+1` or `-1` once known for a receiver and counter.

In [ ]:
out = apply_qerr(tic, qerr)
m = out.meta
print(f"paired {m['matched']} samples, sign {m['qerr_sign']:+d} ({m['sign_choice']})")
print(f"sample-to-sample rms: {fmt(m['diff_rms_before'])} -> {fmt(m['diff_rms_after'])}")

## Stability before and after

TDEV of the PPS against the OCXO. The sawtooth dominates at short averaging times; after the
correction the white noise of the GNSS solution is left, and at long averaging times both curves
meet the OCXO's own wander, which `qErr` does not touch.

In [ ]:
before = nt.compute(out.extra["uncorrected"], 1.0, "tdev")
after = nt.compute(out.offset, 1.0, "tdev")
print(f"{'tau':>6} {'TDEV before':>12} {'after':>10}")
for tau, b, a in zip(before.taus, before.dev, after.dev):
    print(f"{tau:>5g}s {fmt(b):>12} {fmt(a):>10}")

## What the receiver says about itself

`UBX-NAV-CLOCK` gives the receiver's clock bias and drift and its own accuracy estimates. The drift
(here about +270 ns/s) is what makes the sawtooth move; `tAcc` is the receiver's estimate of its time
error, which the TIC measurement can be checked against.

In [ ]:
clk = receiver["UBX-NAV-CLOCK"]
print(f"clock drift: median {np.median(clk.extra['clock_drift']) * 1e9:.1f} ns/s")
print(f"time accuracy estimate tAcc: median {fmt(np.median(clk.extra['time_accuracy']))}")
print(f"corrected PPS spread (detrended rms): {fmt(nt.summary(out)['residual_rms'])}")

## On your own receiver

```bash
ntpstats sawtooth tic.csv receiver.ubx -o corrected.csv   # prints the sign and TDEV before/after
ntpstats stability corrected.csv -k tdev,mtie
```

Enable `UBX-TIM-TP` (and optionally `UBX-NAV-CLOCK`, `UBX-NAV-TIMEUTC`) on the receiver, log its
serial output to a file, and record the TIC readings with POSIX timestamps over the same period.